# SHQ Analysis Program 3 - Isokinetic Baseline Offset Correction
Updated 04/28/2026

-----
## Cell 1 - Imports
Import the necessary packages to run the program.

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from tkinter import Tk
from tkinter.filedialog import askdirectory
from scipy.signal import butter, filtfilt

print(f'{datetime.now()} - Imports OK')

custom_theme = {
    "axes.spines.right":  False,
    "axes.spines.top":    False,
    "axes.titlelocation": "left",
    "axes.titley":        1,
    "font.weight":        "bold",
    "axes.titlesize":     "x-large",
    "axes.labelsize":     "x-large",
    "axes.titleweight":   "bold",
    "axes.labelweight":   "bold"
}
plt.rcParams.update({
    **custom_theme,
    'figure.facecolor': 'white',
    'axes.labelcolor':  'black',
    'axes.edgecolor':   'black',
    'xtick.color':      'black',
    'ytick.color':      'black',
    'axes.titlecolor':  'black'
})

2026-04-28 11:55:48.409477 - Imports OK


-----
## Cell 2 - Define Functions

`butter_lowpass` -- 4th order zero-lag Butterworth filter. 

`pick_directory` -- folder browser dialog.

`get_corrected_name` -- appends `_corrected` to the filename stem (e.g. `LEFT_60deg.csv` → `LEFT_60deg_corrected.csv`).

In [4]:
sf = 2000   # sample frequency (Hz)
fc = 10     # lowpass cutoff (Hz)

def butter_lowpass(torque, cutoff=fc, fs=sf, order=2):
    """
    order=2 with filtfilt gives effective 4th order zero-lag filter.
    """
    b, a = butter(order, cutoff / (fs / 2.0), btype='low')
    return filtfilt(b, a, torque, axis=0)


def pick_directory(prompt="Select Folder"):
    root = Tk()
    root.attributes('-topmost', True)
    root.after(1, root.attributes, '-topmost', False)
    folder = askdirectory(title=prompt)
    root.destroy()
    return folder


def get_corrected_name(filepath):
    """
    Returns the full corrected output path in the same folder.
    e.g. /path/to/LEFT_60deg.csv -> /path/to/LEFT_60deg_corrected.csv
    """
    folder    = os.path.dirname(filepath)
    stem      = os.path.splitext(os.path.basename(filepath))[0]
    return os.path.join(folder, f'{stem}_corrected.csv')


print(f'{datetime.now()} - Functions defined OK')

2026-04-28 11:56:56.388556 - Functions defined OK


-----
## Cell 3 - Select participant folder and load isokinetic files

A dialog will open, navigate to the participant's folder (e.g. `SHQ001`). The code will find all isokinetic files matching the naming convention (`LEFT_60deg`, `RIGHT_60deg`, `LEFT_180deg`, `RIGHT_180deg`) inside that participant's `Full Data` subfolder.

Files that have already been corrected (`_corrected.csv`) are automatically excluded.

In [51]:
participant_folder = pick_directory(prompt="Select the Participant's Data Folder")
participant_id = os.path.basename(participant_folder)
iso_folder = os.path.join(participant_folder, 'Full Data')

# Find all isokinetic files, exclude already-corrected ones
all_files = os.listdir(iso_folder)
iso_files = [
    f for f in sorted(all_files)
    if f.endswith('.csv')
    and '_corrected' not in f
    and any(k in f for k in ['60deg', '180deg'])
]

print(f'Participant: {participant_id}')
print(f'Folder: {iso_folder}')
print('Isokinetic files found:')
for f in iso_files:
    print(f'    - {f[:-4]}')

# Load all into a dict: {filename_stem: (torque_array, full_path)}
iso_dict = {}
for fname in iso_files:
    fpath = os.path.join(iso_folder, fname) # fpath = file path
    df = pd.read_csv(fpath) # df = dataframe of the file path
    torque = df['Torque'].values # read in only the torque column
    torque = torque[~np.isnan(torque)]  # strip any trailing NANs that may have been added
    stem = os.path.splitext(fname)[0] # split extension, same as [:-4]
    iso_dict[stem] = {'torque': torque, 'path': fpath, 'df': df}

print(f'{datetime.now()} - {len(iso_dict)} files loaded OK')

Participant: SHQ022
Folder: E:/Research/0-Mentoring/MoosbruggerJ/Data Collection/SHQ022\Full Data
Isokinetic files found:
    - LEFT_180deg
    - LEFT_60deg
    - RIGHT_180deg
    - RIGHT_60deg
2026-04-28 13:16:24.597200 - 4 files loaded OK


-----
## Cell 4 - Interactive baseline correction

A PyQt6 window will open showing each isokinetic file one at a time.

1. Use the **Baseline Start** and **Baseline End** sliders to select a quiet period before the repetitions begin.
2. Click **Apply Offset** to subtract the baseline mean and preview the corrected + filtered signal below.
3. If the correction looks correct, click **Save & Next** to write the corrected CSV to the same folder and advance.
4. If not happy, adjust sliders and re-apply.
5. Click **Reset Sliders** to return to defaults.

The corrected file is saved as `LEFT_60deg_corrected.csv` (same folder, `_corrected` appended to the name).


In [52]:
import sys
import qdarktheme
import matplotlib
matplotlib.use('QtAgg')
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator
from PyQt6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout,
                              QHBoxLayout, QLabel, QPushButton, QSlider,
                              QProgressBar, QSizePolicy)
from PyQt6.QtCore import Qt
from PyQt6.QtGui import QFont

plt.rcParams.update({
    **custom_theme,
    'figure.facecolor': 'none',
    'axes.facecolor':   'none',
    'axes.labelcolor':  '#ffffff',
    'axes.edgecolor':   '#ffffff',
    'xtick.color':      '#ffffff',
    'ytick.color':      '#ffffff',
    'axes.titlecolor':  'white',
    'legend.labelcolor':'white',
    'legend.facecolor': 'none',
    'legend.edgecolor': 'none',
})

saved_files = []   # populated as files are saved
file_names  = list(iso_dict.keys())

# Style constants
BTN_PRIMARY = """
    QPushButton {
        background-color: #4caf50; color: white; font-weight: bold;
        font-size: 13px; padding: 8px 20px; border-radius: 4px; border: none;
    }
    QPushButton:hover    { background-color: #43a047; }
    QPushButton:pressed  { background-color: #388e3c; }
    QPushButton:disabled { background-color: #555; color: #888; }
"""
BTN_APPLY = """
    QPushButton {
        background-color: #2196f3; color: white; font-weight: bold;
        font-size: 13px; padding: 8px 20px; border-radius: 4px; border: none;
    }
    QPushButton:hover    { background-color: #1e88e5; }
    QPushButton:pressed  { background-color: #1565c0; }
    QPushButton:disabled { background-color: #555; color: #888; }
"""
BTN_RESET = """
    QPushButton {
        background-color: #e67e22; color: white; font-weight: bold;
        font-size: 13px; padding: 8px 20px; border-radius: 4px; border: none;
    }
    QPushButton:hover   { background-color: #d35400; }
    QPushButton:pressed { background-color: #ba4a00; }
"""
LABEL_STYLE  = "color: white; font-size: 13px;"
STATUS_STYLE = "color: #aaaaaa; font-size: 12px;"
SAVED_STYLE  = "color: #4caf50; font-size: 13px; font-weight: bold;"
PROG_STYLE   = """
    QProgressBar {
        border: 1px solid #555; border-radius: 4px;
        text-align: center; color: white; background: #2b2b2b;
    }
    QProgressBar::chunk { background-color: #4caf50; border-radius: 3px; }
"""

class IsokineticsWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.i            = 0
        self.file_name    = None
        self.torque_raw   = None
        self.duration     = None
        self.current_base = None
        self.current_x0   = None
        self.current_x1   = None

        self.setWindowTitle('SHQ Isokinetic Baseline Correction')
        self.setMinimumSize(1200, 800)
        self.showMaximized()

        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)
        layout.setSpacing(4)
        layout.setContentsMargins(16, 10, 16, 10)

        # File label
        self.file_label = QLabel('')
        self.file_label.setFont(QFont('Arial', 14, QFont.Weight.Bold))
        self.file_label.setStyleSheet(LABEL_STYLE)
        layout.addWidget(self.file_label)

        # Canvas 1 — raw signal
        self.fig1 = Figure()
        self.ax1 = self.fig1.add_subplot(111)
        self.canvas1 = FigureCanvas(self.fig1)
        self.canvas1.setSizePolicy(QSizePolicy.Policy.Expanding,
                                   QSizePolicy.Policy.Expanding)
        layout.addWidget(self.canvas1, stretch=4)

        # Sliders 
        for attr, lbl_attr, lbl_text in [
            ('sld_start', 'lbl_start', 'Baseline Start:'),
            ('sld_end',   'lbl_end',   'Baseline End:  '),
        ]:
            row = QHBoxLayout()
            lbl = QLabel(lbl_text)
            lbl.setFont(QFont('Arial', 10))
            lbl.setStyleSheet(LABEL_STYLE)
            lbl.setFixedWidth(140)
            setattr(self, lbl_attr, lbl)

            sld = QSlider(Qt.Orientation.Horizontal)
            sld.setMinimum(0)
            sld.setMaximum(10000)
            sld.valueChanged.connect(self.on_slider)
            setattr(self, attr, sld)

            val_lbl = QLabel('0.000 s')
            val_lbl.setFont(QFont('Arial', 10))
            val_lbl.setStyleSheet(LABEL_STYLE)
            val_lbl.setFixedWidth(75)
            setattr(self, lbl_attr + '_val', val_lbl)

            row.addWidget(lbl)
            row.addWidget(sld)
            row.addWidget(val_lbl)
            layout.addLayout(row)

        # Status label 
        self.status_label = QLabel('')
        self.status_label.setStyleSheet(STATUS_STYLE)
        self.status_label.setFont(QFont('Arial', 11))
        layout.addWidget(self.status_label)

        # Buttons
        btn_row = QHBoxLayout()
        btn_row.setSpacing(12)

        self.btn_reset = QPushButton('Reset Sliders')
        self.btn_reset.setStyleSheet(BTN_RESET)
        self.btn_reset.clicked.connect(self.on_reset)

        self.btn_apply = QPushButton('Apply Offset')
        self.btn_apply.setStyleSheet(BTN_APPLY)
        self.btn_apply.clicked.connect(self.on_apply)

        self.btn_save = QPushButton('Save | Next')
        self.btn_save.setStyleSheet(BTN_PRIMARY)
        self.btn_save.setEnabled(False)
        self.btn_save.clicked.connect(self.on_save)

        btn_row.addWidget(self.btn_reset)
        btn_row.addWidget(self.btn_apply)
        btn_row.addWidget(self.btn_save)
        btn_row.addStretch()
        layout.addLayout(btn_row)

        # Canvas 2 — corrected torque signal
        self.fig2 = Figure()
        self.ax2 = self.fig2.add_subplot(111)
        self.canvas2 = FigureCanvas(self.fig2)
        self.canvas2.setSizePolicy(QSizePolicy.Policy.Expanding,
                                   QSizePolicy.Policy.Expanding)
        layout.addWidget(self.canvas2, stretch=4)

        # Saved label 
        self.saved_label = QLabel('')
        self.saved_label.setStyleSheet(SAVED_STYLE)
        self.saved_label.setFont(QFont('Arial', 11, QFont.Weight.Bold))
        layout.addWidget(self.saved_label)

        # Progress bar 
        self.progress = QProgressBar()
        self.progress.setMinimum(0)
        self.progress.setMaximum(len(file_names))
        self.progress.setValue(0)
        self.progress.setFixedHeight(22)
        self.progress.setStyleSheet(PROG_STYLE)
        self.progress.setFormat(f'%v / {len(file_names)} files')
        layout.addWidget(self.progress)
        self.load_file(0)

    # Load file
    def load_file(self, i):
        self.file_name = file_names[i]
        self.torque_raw = iso_dict[self.file_name]['torque']
        self.t = np.arange(len(self.torque_raw)) / sf
        self.duration = self.t[-1]
        self.current_base = None
        self.current_x0 = None
        self.current_x1 = None

        max_ms = int(self.duration * 1000)
        default_end = int(min(500, max_ms * 0.05))

        self.sld_start.valueChanged.disconnect()
        self.sld_end.valueChanged.disconnect()
        self.sld_start.setMaximum(max_ms)
        self.sld_end.setMaximum(max_ms)
        self.sld_start.setValue(0)
        self.sld_end.setValue(default_end)
        self.sld_start.valueChanged.connect(self.on_slider)
        self.sld_end.valueChanged.connect(self.on_slider)

        self.file_label.setText(
            f'File {i+1} / {len(file_names)}  —  {self.file_name}'
        )
        self.saved_label.setText('')
        self.lbl_start_val.setText('0.000 s')
        self.lbl_end_val.setText(f'{default_end/1000:.3f} s')
        self.btn_save.setEnabled(False)

        self.ax2.clear()
        self.canvas2.draw()

        self.redraw_raw(0.0, default_end / 1000)

    # Redraw raw signal
    def redraw_raw(self, x0, x1):
        t = self.t
        tor = self.torque_raw
        i0 = np.searchsorted(t, x0)
        i1 = np.searchsorted(t, x1)
        seg = tor[i0:i1]
        baseline_val = float(seg.mean()) if len(seg) > 0 else 0.0

        self.ax1.clear()
        self.ax1.plot(t, tor, color='#5b9bd5', lw=0.7, label='Raw torque')
        self.ax1.axhline(0, color='#666666', lw=0.5, ls='--')
        self.ax1.axvline(x0, color='#5b9bd5', lw=2, ls='--',
                         label=f'Start: {x0:.3f} s')
        self.ax1.axvline(x1, color='#e67e22', lw=2, ls='--',
                         label=f'End: {x1:.3f} s')
        self.ax1.axvspan(x0, x1, alpha=0.18, facecolor='#5b9bd5')
        self.ax1.axhline(baseline_val, color='#e74c3c', lw=1.2, ls=':',
                         label=f'Baseline mean: {baseline_val:.4f} Nm')
        self.ax1.set_xlabel('Time (s)')
        self.ax1.set_ylabel('Torque (Nm)')
        self.ax1.set_title(f'Raw Signal {participant_id} | {self.file_name}', fontsize=12)
        self.ax1.spines['top'].set_visible(False)
        self.ax1.spines['right'].set_visible(False)
        self.ax1.xaxis.set_major_locator(MaxNLocator(integer=False, prune='both'))
        self.ax1.yaxis.set_major_locator(MaxNLocator(nbins='auto', prune='both'))
        self.fig1.tight_layout(pad=1.5)
        self.canvas1.draw()

        self.status_label.setText(
            f'Window: {x0:.3f} – {x1:.3f} s  '
            f'({(x1-x0)*1000:.0f} ms)  |  '
            f'Baseline mean: {baseline_val:.4f} N·m'
        )

    # Draw corrected signal
    def draw_corrected(self, torque_corr, torque_filt, baseline_val):
        t = self.t
        self.ax2.clear()
        self.ax2.plot(t, torque_corr, color='#5b9bd5', lw=0.5,
                      ls='--', label='Offset corrected')
        self.ax2.plot(t, torque_filt, color='white', lw=1.8,
                      label='Filtered (glimpse)')
        self.ax2.axhline(0, color='#666666', lw=0.5, ls='--')
        self.ax2.set_xlabel('Time (s)')
        self.ax2.set_ylabel('Corrected Torque (Nm)')
        self.ax2.set_title(
            f'Offset Corrected = {self.file_name}  '
            f'(Baseline = {baseline_val:.4f} Nm)',
            fontsize=12
        )
        self.ax2.spines['top'].set_visible(False)
        self.ax2.spines['right'].set_visible(False)
        self.ax2.xaxis.set_major_locator(MaxNLocator(integer=False, prune='both'))
        self.ax2.yaxis.set_major_locator(MaxNLocator(nbins='auto', prune='both'))
        self.ax2.legend(fontsize=9, loc='upper right', frameon=False)
        self.fig2.tight_layout(pad=1.5)
        self.canvas2.draw()

    # Slider changed 
    def on_slider(self):
        x0 = self.sld_start.value() / 1000
        x1 = self.sld_end.value()   / 1000
        self.lbl_start_val.setText(f'{x0:.3f} s')
        self.lbl_end_val.setText(  f'{x1:.3f} s')
        if x0 >= x1:
            self.status_label.setText('⚠  Start must be before End.')
            return
        self.btn_save.setEnabled(False)
        self.saved_label.setText('')
        self.redraw_raw(x0, x1)

    # Reset 
    def on_reset(self):
        max_ms = int(self.duration * 1000)
        default_end = int(min(500, max_ms * 0.05))
        self.sld_start.setValue(0)
        self.sld_end.setValue(default_end)

    # Apply offset 
    def on_apply(self):
        x0 = self.sld_start.value() / 1000
        x1 = self.sld_end.value()   / 1000

        if x0 >= x1:
            self.status_label.setText('⚠  Fix slider positions first.')
            return

        t = self.t
        tor = self.torque_raw
        i0 = np.searchsorted(t, x0)
        i1 = np.searchsorted(t, x1)

        baseline_val = float(tor[i0:i1].mean())
        torque_corr = tor - baseline_val
        torque_filt = butter_lowpass(torque_corr)
        self.current_base = baseline_val
        self.current_x0 = x0
        self.current_x1 = x1
        self.torque_corr = torque_corr   # store for saving

        self.draw_corrected(torque_corr, torque_filt, baseline_val)

        self.saved_label.setText(
            f'Baseline = {baseline_val:.4f} Nm  —  '
            f'Click Save & Next.'
        )
        self.btn_save.setEnabled(True)

    # Save & next
    def on_save(self):
        # Build corrected dataframe with only Torque
        n = len(self.torque_corr)
        new_dat = pd.DataFrame({'Torque' : self.torque_corr[:n]})

        # Save to same folder with _corrected suffix
        out_path = get_corrected_name(iso_dict[self.file_name]['path'])
        new_dat.to_csv(out_path, index=False)
        saved_files.append(out_path)

        self.saved_label.setText(
            f'Saved: {os.path.basename(out_path)}'
        )

        self.i += 1
        self.progress.setValue(self.i)

        if self.i < len(file_names):
            self.load_file(self.i)
        else:
            self.file_label.setText(
                'All files corrected and saved — close this window.'
            )
            self.btn_apply.setEnabled(False)
            self.btn_save.setEnabled(False)

    # Resize event
    def resizeEvent(self, event):
        super().resizeEvent(event)
        try:
            if self.torque_raw is not None:
                x0 = self.sld_start.value() / 1000
                x1 = self.sld_end.value()   / 1000
                if x0 < x1:
                    self.redraw_raw(x0, x1)
        except Exception:
            pass


# Launch
app = QApplication.instance() or QApplication(sys.argv)
app.setStyleSheet(qdarktheme.load_stylesheet())
win = IsokineticsWindow()
win.show()
app.exec()

print(f'Done, {len(saved_files)} files saved for {participant_id}:')
for p in saved_files:
    print(f'  {os.path.basename(p)[:-4]}')

Done, 4 files saved for SHQ022:
  LEFT_180deg_corrected
  LEFT_60deg_corrected
  RIGHT_180deg_corrected
  RIGHT_60deg_corrected
